# Visualizations of loss Function

In [1]:
import sys
import os
main_dir = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(main_dir)

import argparse
import os
import time

import numpy as np

from MambaFCS.changedetection.configs.config import get_config

import torch
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
from MambaFCS.changedetection.datasets.make_data_loader import SemanticChangeDetectionDatset, make_data_loader
from MambaFCS.changedetection.utils_func.metrics import Evaluator
from MambaFCS.changedetection.models.STMambaSCD import STMambaSCD
import MambaFCS.changedetection.utils_func.lovasz_loss as L
from torch.optim.lr_scheduler import StepLR

from MambaFCS.changedetection.utils_func.mcd_utils import accuracy, SCDD_eval_all, AverageMeter

/home/buddhiw/miniconda3/envs/Mamba/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/buddhiw/ChangeDetection/CDMamba/MambaFCS/classification/models/vmamba.py:252: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/home/buddhiw/ChangeDetection/CDMamba/MambaFCS/classification/models/vmamba.py:260: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd
/home/buddhiw/ChangeDetection/CDMamba/MambaFCS/classification/models/vmamba.py:275: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @

In [2]:
torch.cuda.set_device(1)

In [3]:
MODEL_PATH = os.path.join(
    main_dir, "MambaFCS/saved_models/CA_spatial_2/32000_model_0.244.pth"
)

In [4]:
from dotenv import load_dotenv

load_dotenv()
def getPath(env_path):
    return os.path.expanduser(os.getenv(env_path))

VSSM_MODEL_PATH = getPath('VSSMBASEPATH')

SECOND_DATASET_PATH = getPath('SECONDDATASETPATH')

SECOND_TRAIN_DATASET_PATH = os.path.join(SECOND_DATASET_PATH, 'train')
SECOND_TEST_DATASET_PATH = os.path.join(SECOND_DATASET_PATH, 'test')
SECOND_TRAIN_DATA_LIST_PATH = os.path.join(SECOND_DATASET_PATH, 'train.txt')
SECOND_TEST_DATA_LIST_PATH = os.path.join(SECOND_DATASET_PATH, 'test.txt')

print(SECOND_DATASET_PATH)

/home/buddhiw/ChangeDetection/Datasets/SECOND


In [5]:
configs_path = os.path.join(main_dir, 'MambaFCS/changedetection/configs/vssm1/vssm_base_224.yaml')
model_path = os.path.join(main_dir, 'MambaFCS/saved_models')


train_data_list = []
with open(SECOND_TRAIN_DATA_LIST_PATH, 'r') as f:
    for line in f:
        train_data_list.append(line.strip())

test_data_list = []
with open(SECOND_TEST_DATA_LIST_PATH, 'r') as f:
    for line in f:
        test_data_list.append(line.strip())

class ARGS:
    def __init__(self):
        self.cfg = configs_path
        self.opts = None
        self.pretrained_weight_path = VSSM_MODEL_PATH
        self.dataset = 'SECOND'
        self.type = 'train'
        self.train_dataset_path = SECOND_TRAIN_DATASET_PATH

        self.test_dataset_path = SECOND_TEST_DATASET_PATH

        
        self.shuffle = True
        self.batch_size = 4
        self.crop_size = 256
        self.train_data_name_list = train_data_list
        self.test_data_name_list = test_data_list
        self.start_iter = 0
        self.cuda = True
        self.max_iters = 800000
        self.model_type = 'MambaSCD_Base'
        self.model_param_path = model_path
        self.resume = MODEL_PATH
        self.learning_rate = 1e-4
        self.momentum = 0.9
        self.weight_decay = 5e-4

args = ARGS()

In [6]:
args = args
config = get_config(args)

train_data_loader = make_data_loader(args)

deep_model = STMambaSCD(
    output_cd = 2, 
    output_clf = 7,
    pretrained=args.pretrained_weight_path,
    patch_size=config.MODEL.VSSM.PATCH_SIZE, 
    in_chans=config.MODEL.VSSM.IN_CHANS, 
    num_classes=config.MODEL.NUM_CLASSES, 
    depths=config.MODEL.VSSM.DEPTHS, 
    dims=config.MODEL.VSSM.EMBED_DIM, 
    # ===================
    ssm_d_state=config.MODEL.VSSM.SSM_D_STATE,
    ssm_ratio=config.MODEL.VSSM.SSM_RATIO,
    ssm_rank_ratio=config.MODEL.VSSM.SSM_RANK_RATIO,
    ssm_dt_rank=("auto" if config.MODEL.VSSM.SSM_DT_RANK == "auto" else int(config.MODEL.VSSM.SSM_DT_RANK)),
    ssm_act_layer=config.MODEL.VSSM.SSM_ACT_LAYER,
    ssm_conv=config.MODEL.VSSM.SSM_CONV,
    ssm_conv_bias=config.MODEL.VSSM.SSM_CONV_BIAS,
    ssm_drop_rate=config.MODEL.VSSM.SSM_DROP_RATE,
    ssm_init=config.MODEL.VSSM.SSM_INIT,
    forward_type=config.MODEL.VSSM.SSM_FORWARDTYPE,
    # ===================
    mlp_ratio=config.MODEL.VSSM.MLP_RATIO,
    mlp_act_layer=config.MODEL.VSSM.MLP_ACT_LAYER,
    mlp_drop_rate=config.MODEL.VSSM.MLP_DROP_RATE,
    # ===================
    drop_path_rate=config.MODEL.DROP_PATH_RATE,
    patch_norm=config.MODEL.VSSM.PATCH_NORM,
    norm_layer=config.MODEL.VSSM.NORM_LAYER,
    downsample_version=config.MODEL.VSSM.DOWNSAMPLE,
    patchembed_version=config.MODEL.VSSM.PATCHEMBED,
    gmlp=config.MODEL.VSSM.GMLP,
    use_checkpoint=config.TRAIN.USE_CHECKPOINT,
    ) 

deep_model.load_state_dict(torch.load(MODEL_PATH))

=> merge config from /home/buddhiw/ChangeDetection/CDMamba/MambaFCS/changedetection/configs/vssm1/vssm_base_224.yaml
Successfully load ckpt /home/buddhiw/ChangeDetection/VSSModels/pretrained/vssm_base_0229_ckpt_epoch_237.pth
_IncompatibleKeys(missing_keys=['outnorm0.weight', 'outnorm0.bias', 'outnorm1.weight', 'outnorm1.bias', 'outnorm2.weight', 'outnorm2.bias', 'outnorm3.weight', 'outnorm3.bias'], unexpected_keys=['classifier.norm.weight', 'classifier.norm.bias', 'classifier.head.weight', 'classifier.head.bias'])
False


/home/buddhiw/miniconda3/envs/Mamba/lib/python3.13/site-packages/torch/nn/init.py:511: UserWarning: Initializing zero-element tensors is a no-op
  warnings.warn("Initializing zero-element tensors is a no-op")


<All keys matched successfully>

# Validation

In [9]:
deep_model = deep_model.cuda()  # Move the model to GPU

deep_model.eval()  # Set the model to evaluation mode


dataset = SemanticChangeDetectionDatset(args.test_dataset_path, args.test_data_name_list, 256, None, 'test')
val_data_loader = DataLoader(dataset, batch_size=1, num_workers=4, drop_last=False)
torch.cuda.empty_cache()
acc_meter = AverageMeter()

preds_all = []
labels_all = []
with torch.no_grad():
    for itera, data in enumerate(val_data_loader):
        pre_change_imgs, post_change_imgs, labels_cd, labels_clf_t1, labels_clf_t2, _ = data

        pre_change_imgs = pre_change_imgs.cuda()
        post_change_imgs = post_change_imgs.cuda()
        labels_cd = labels_cd.cuda().long()
        labels_clf_t1 = labels_clf_t1.cuda().long()
        labels_clf_t2 = labels_clf_t2.cuda().long()


        # input_data = torch.cat([pre_change_imgs, post_change_imgs], dim=1)
        output_1, output_semantic_t1, output_semantic_t2 = deep_model(pre_change_imgs, post_change_imgs)

        labels_cd = labels_cd.cpu().numpy()
        labels_A = labels_clf_t1.cpu().numpy()
        labels_B = labels_clf_t2.cpu().numpy()

        change_mask = torch.argmax(output_1, axis=1).cpu().numpy()

        preds_A = torch.argmax(output_semantic_t1, dim=1).cpu().numpy()
        preds_B = torch.argmax(output_semantic_t2, dim=1).cpu().numpy()

        # preds_A[change_mask == 0] = 0
        # preds_B[change_mask == 0] = 0

        if itera % 100 == 0:
            print(f'iter is {itera}')

        preds_scd = (preds_A - 1) * 6 + preds_B
        preds_scd[change_mask == 0] = 0

        labels_scd = (labels_A - 1) * 6 + labels_B
        labels_scd[labels_cd == 0] = 0

        for (pred_scd, label_scd) in zip(preds_scd, labels_scd):
            acc_A, valid_sum_A = accuracy(pred_scd, label_scd)
            preds_all.append(pred_scd)
            labels_all.append(label_scd)
            acc = acc_A
            acc_meter.update(acc)


kappa_n0, Fscd, IoU_mean, Sek = SCDD_eval_all(preds_all, labels_all, 37)
print(f'Kappa coefficient rate is {kappa_n0}, F1 is {Fscd}, OA is {acc_meter.avg}, '
        f'mIoU is {IoU_mean}, SeK is {Sek}')


iter is 0
iter is 100
iter is 200
iter is 300
iter is 400
iter is 500
iter is 600
iter is 700
iter is 800
iter is 900
iter is 1000
iter is 1100
iter is 1200
iter is 1300
iter is 1400
iter is 1500
iter is 1600
Kappa coefficient rate is 0.35850256528877217, F1 is 0.5718850643976622, OA is 0.8672652351532522, mIoU is 0.7338657055021071, SeK is 0.23676391146202122
